In [ ]:
# ============================================================
# STEP 1: IMPORT LIBRARIES AND INITIALIZE AI POSE DETECTOR
# ============================================================

import cv2
import mediapipe as mp
import numpy as np
import time
import math
import os
import urllib.request

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("Libraries imported successfully!")

# ============================================================
# DOWNLOAD MEDIAPIPE POSE MODEL
# ============================================================

MODEL_PATH = "pose_landmarker_lite.task"

if not os.path.exists(MODEL_PATH):

    url = (
        "https://storage.googleapis.com/mediapipe-models/"
        "pose_landmarker/pose_landmarker_lite/float16/1/"
        "pose_landmarker_lite.task"
    )

    urllib.request.urlretrieve(url, MODEL_PATH)

print("Pose model ready!")

# ============================================================
# INITIALIZE POSE LANDMARKER
# ============================================================

options = vision.PoseLandmarkerOptions(

    base_options=python.BaseOptions(
        model_asset_path=MODEL_PATH
    ),

    running_mode=vision.RunningMode.VIDEO,

    num_poses=1,

    min_pose_detection_confidence=0.5,

    min_pose_presence_confidence=0.5,

    min_tracking_confidence=0.5
)

landmarker = vision.PoseLandmarker.create_from_options(options)

print("==========================================")
print("AI POSE DETECTOR INITIALIZED SUCCESSFULLY")
print("==========================================")

Libraries imported successfully!
Pose model ready!
AI POSE DETECTOR INITIALIZED SUCCESSFULLY


In [ ]:
# ============================================================
# STEP 2: AI DETECTION FUNCTIONS
# ============================================================

# ============================================================
# 1. CALCULATE JOINT ANGLE
# ============================================================

def calculate_angle(a, b, c):

    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = (
        np.arctan2(c[1] - b[1], c[0] - b[0])
        -
        np.arctan2(a[1] - b[1], a[0] - b[0])
    )

    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180:
        angle = 360 - angle

    return angle


# ============================================================
# 2. SKELETON CONNECTIONS
# ============================================================

connections = [

    (11, 12),

    (11, 13),
    (13, 15),

    (12, 14),
    (14, 16),

    (11, 23),
    (12, 24),

    (23, 24),

    (23, 25),
    (25, 27),

    (24, 26),
    (26, 28)

]


# ============================================================
# 3. DRAW BODY SKELETON
# ============================================================

def draw_skeleton(frame, landmarks):

    h, w, _ = frame.shape

    # Draw connecting lines

    for start, end in connections:

        p1 = landmarks[start]
        p2 = landmarks[end]

        if min(p1.visibility, p2.visibility) > 0.5:

            x1 = int(p1.x * w)
            y1 = int(p1.y * h)

            x2 = int(p2.x * w)
            y2 = int(p2.y * h)

            cv2.line(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                3
            )

    # Draw landmarks

    for point in landmarks:

        if point.visibility > 0.5:

            x = int(point.x * w)
            y = int(point.y * h)

            cv2.circle(
                frame,
                (x, y),
                5,
                (0, 0, 255),
                -1
            )

    return frame


print("Angle calculation function ready!")
print("Skeleton tracking function ready!")
print("STEP 2 COMPLETED!")

Angle calculation function ready!
Skeleton tracking function ready!
STEP 2 COMPLETED!


In [ ]:
# ============================================================
# STEP 3: COMPLETE AI WORKOUT REP COUNTER
# ============================================================

# ============================================================
# 1. CAMERA SETUP
# ============================================================

cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_AUTOFOCUS, 1)

if not cap.isOpened():

    raise RuntimeError("Camera not detected!")

WINDOW_NAME = "AI WORKOUT REP COUNTER"

cv2.namedWindow(
    WINDOW_NAME,
    cv2.WINDOW_NORMAL
)

cv2.resizeWindow(
    WINDOW_NAME,
    1400,
    800
)


# ============================================================
# 2. DISPLAY SETTINGS
# ============================================================

CAMERA_WIDTH = 1000
CAMERA_HEIGHT = 563

PANEL_WIDTH = 280


# ============================================================
# 3. EXERCISE SETTINGS
# ============================================================

exercise = "BICEP CURLS"

reps = {

    "BICEP CURLS": 0,

    "SQUATS": 0,

    "SHOULDER PRESS": 0

}

stage = "UP"

start_time = time.time()

frame_timestamp = 0

angle = 0

body_detected = False


# ============================================================
# 4. DASHBOARD FUNCTION
# ============================================================

def draw_dashboard():

    panel = np.zeros(
        (CAMERA_HEIGHT, PANEL_WIDTH, 3),
        dtype=np.uint8
    )

    panel[:] = (25, 25, 25)

    # Dashboard title

    cv2.putText(
        panel,
        "AI FITNESS TRACKER",
        (15, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 255),
        2
    )

    cv2.line(
        panel,
        (15, 50),
        (PANEL_WIDTH - 15, 50),
        (100, 100, 100),
        1
    )

    # Exercise

    cv2.putText(
        panel,
        "CURRENT EXERCISE",
        (15, 85),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        (180, 180, 180),
        1
    )

    cv2.putText(
        panel,
        exercise,
        (15, 115),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.48,
        (255, 255, 255),
        2
    )

    # Repetitions

    cv2.putText(
        panel,
        "REPETITIONS",
        (15, 165),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        (180, 180, 180),
        1
    )

    cv2.putText(
        panel,
        str(reps[exercise]),
        (15, 230),
        cv2.FONT_HERSHEY_SIMPLEX,
        2.0,
        (0, 255, 0),
        4
    )

    # Stage

    cv2.putText(
        panel,
        "STAGE",
        (15, 280),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        (180, 180, 180),
        1
    )

    cv2.putText(
        panel,
        stage,
        (15, 320),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.85,
        (0, 255, 255),
        2
    )

    # Joint angle

    cv2.putText(
        panel,
        f"ANGLE: {int(angle)} deg",
        (15, 370),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.50,
        (255, 255, 255),
        1
    )

    # Timer

    elapsed = int(time.time() - start_time)

    minutes = elapsed // 60
    seconds = elapsed % 60

    cv2.putText(
        panel,
        f"TIME: {minutes:02d}:{seconds:02d}",
        (15, 415),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.50,
        (255, 255, 255),
        1
    )

    # Detection status

    if body_detected:

        status = "BODY DETECTED"
        color = (0, 255, 0)

    else:

        status = "NO BODY DETECTED"
        color = (0, 0, 255)

    cv2.putText(
        panel,
        status,
        (15, 465),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        color,
        1
    )

    # Controls

    cv2.line(
        panel,
        (15, 485),
        (PANEL_WIDTH - 15, 485),
        (100, 100, 100),
        1
    )

    cv2.putText(
        panel,
        "WORKOUT CONTROLS",
        (15, 510),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.42,
        (0, 255, 255),
        1
    )

    cv2.putText(
        panel,
        "1 - BICEP CURLS",
        (15, 535),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.36,
        (255, 255, 255),
        1
    )

    cv2.putText(
        panel,
        "2 - SQUATS",
        (15, 553),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.36,
        (255, 255, 255),
        1
    )

    return panel


# ============================================================
# 5. START WORKOUT
# ============================================================

print("==========================================")
print("AI WORKOUT REP COUNTER STARTED")
print("==========================================")

print("1 - BICEP CURLS")
print("2 - SQUATS")
print("3 - SHOULDER PRESS")
print("R - RESET")
print("Q - EXIT")


# ============================================================
# 6. MAIN DETECTION LOOP
# ============================================================

try:

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:

            print("Camera frame not received!")
            break

        # Mirror camera

        frame = cv2.flip(frame, 1)

        h, w, _ = frame.shape

        # Convert BGR to RGB

        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        frame_timestamp += 33

        # AI pose detection

        results = landmarker.detect_for_video(
            mp_image,
            frame_timestamp
        )

        body_detected = False
        angle = 0

        # ====================================================
        # 7. BODY DETECTION
        # ====================================================

        if results.pose_landmarks:

            landmarks = results.pose_landmarks[0]

            body_detected = True

            draw_skeleton(
                frame,
                landmarks
            )

            # Body landmarks

            shoulder = [
                landmarks[12].x * w,
                landmarks[12].y * h
            ]

            elbow = [
                landmarks[14].x * w,
                landmarks[14].y * h
            ]

            wrist = [
                landmarks[16].x * w,
                landmarks[16].y * h
            ]

            hip = [
                landmarks[24].x * w,
                landmarks[24].y * h
            ]

            knee = [
                landmarks[26].x * w,
                landmarks[26].y * h
            ]

            ankle = [
                landmarks[28].x * w,
                landmarks[28].y * h
            ]

            # ====================================================
            # 8. BICEP CURLS
            # ====================================================

            if exercise == "BICEP CURLS":

                angle = calculate_angle(
                    shoulder,
                    elbow,
                    wrist
                )

                visibility = min(
                    landmarks[12].visibility,
                    landmarks[14].visibility,
                    landmarks[16].visibility
                )

                if visibility > 0.5:

                    if angle < 60:

                        stage = "DOWN"

                    if angle > 150 and stage == "DOWN":

                        stage = "UP"

                        reps[exercise] += 1

            # ====================================================
            # 9. SQUATS
            # ====================================================

            elif exercise == "SQUATS":

                angle = calculate_angle(
                    hip,
                    knee,
                    ankle
                )

                visibility = min(
                    landmarks[24].visibility,
                    landmarks[26].visibility,
                    landmarks[28].visibility
                )

                if visibility > 0.5:

                    if angle < 90:

                        stage = "DOWN"

                    if angle > 160 and stage == "DOWN":

                        stage = "UP"

                        reps[exercise] += 1

            # ====================================================
            # 10. SHOULDER PRESS
            # ====================================================

            elif exercise == "SHOULDER PRESS":

                angle = calculate_angle(
                    shoulder,
                    elbow,
                    wrist
                )

                visibility = min(
                    landmarks[12].visibility,
                    landmarks[14].visibility,
                    landmarks[16].visibility
                )

                if visibility > 0.5:

                    if angle < 70:

                        stage = "DOWN"

                    if (
                        angle > 150
                        and wrist[1] < shoulder[1]
                        and stage == "DOWN"
                    ):

                        stage = "UP"

                        reps[exercise] += 1

            # Display angle

            cv2.putText(
                frame,
                f"ANGLE: {int(angle)}",
                (25, 45),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2
            )

        else:

            cv2.putText(
                frame,
                "BODY NOT DETECTED",
                (25, 45),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

        # ====================================================
        # 11. CAMERA DISPLAY
        # ====================================================

        camera_display = cv2.resize(
            frame,
            (CAMERA_WIDTH, CAMERA_HEIGHT)
        )

        # ====================================================
        # 12. DASHBOARD
        # ====================================================

        dashboard = draw_dashboard()

        combined = np.hstack(
            (camera_display, dashboard)
        )

        # ====================================================
        # 13. SHOW WINDOW
        # ====================================================

        cv2.imshow(
            WINDOW_NAME,
            combined
        )

        # ====================================================
        # 14. KEYBOARD CONTROLS
        # ====================================================

        key = cv2.waitKey(1) & 0xFF

        if key == ord('1'):

            exercise = "BICEP CURLS"
            stage = "UP"

        elif key == ord('2'):

            exercise = "SQUATS"
            stage = "UP"

        elif key == ord('3'):

            exercise = "SHOULDER PRESS"
            stage = "DOWN"

        elif key == ord('r'):

            reps = {
                "BICEP CURLS": 0,
                "SQUATS": 0,
                "SHOULDER PRESS": 0
            }

            stage = "UP"

            start_time = time.time()

        elif key == ord('q'):

            break


# ============================================================
# 15. STOP WORKOUT
# ============================================================

except KeyboardInterrupt:

    print("Workout interrupted!")

except Exception as e:

    print("Workout error:", e)

finally:

    cap.release()

    if "landmarker" in globals():

        landmarker.close()

    cv2.destroyAllWindows()

    print("\n==========================================")
    print("WORKOUT SUMMARY")
    print("==========================================")

    for name, count in reps.items():

        print(f"{name}: {count} REPS")

    print("==========================================")
    print("THANK YOU FOR USING AI FITNESS TRACKER!")

AI WORKOUT REP COUNTER STARTED
1 - BICEP CURLS
2 - SQUATS
3 - SHOULDER PRESS
R - RESET
Q - EXIT

WORKOUT SUMMARY
BICEP CURLS: 8 REPS
SQUATS: 0 REPS
SHOULDER PRESS: 5 REPS
THANK YOU FOR USING AI FITNESS TRACKER!
